In [ ]:
import os
import sys
import torch
import torch.nn as nn
import torch.optim as optim
import argparse
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt

args = {
      'ckptpath': './checkpoint.pth',
      'dataset_dir': './food/'
}
args = argparse.Namespace(**args) ## 作用是将字典转换为一个对象，使得可以通过点操作符来访问字典中的值。例如，args.ckptpath 将返回 './checkpoint.pth'，而 args.dataset_dir 将返回 './food/'。

class Classifier(nn.Module):
    def __init__(self):
        super(Classifier, self).__init__()
        def build_block(in_dim,out_dim):
            return [
                nn.Conv2d(in_dim,out_dim,3,1,1),
                nn.BatchNorm2d(out_dim),
                nn.ReLU(inplace=True) ## inplace=True 的作用是直接在输入数据上进行操作，节省内存空间。
            ]
        def stack_block(indim,outdim,num):
            layers = build_block(indim,outdim)
            for i in range(num-1):
                layers += build_block(outdim,outdim)
            layers.append(nn.MaxPool2d(2,2))
            return layers
        
        cnn_list = []
        cnn_list += stack_block(3,128,3) ## (128,64,64)
        cnn_list += stack_block(128,128,3) ## (128,32,32)
        cnn_list += stack_block(128,256,3) ## (256,16,16)
        cnn_list += stack_block(256,512,1) ## (512,8,8)
        cnn_list += stack_block(512,512,1) ## (512,4,4)
        self.cnn = nn.Sequential(*cnn_list)
        
        dnn_list = [
            nn.Linear(512*4*4,1024),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(1024,11)
        ]
        self.fc = nn.Sequential(*dnn_list)
        
    def forward(self,x):
        out = self.cnn(x)
        out = out.reshape(out.size(0),-1)
        out = self.fc(out)
        return out
    
model = Classifier().cuda()
checkpoint = torch.load(args.ckptpath)
model.load_state_dict(checkpoint['model_state_dict'])

class FoodDataset(Dataset):
    def __init__(self, paths, labels, mode):  ## paths 是图像文件的路径列表
        self.paths = paths
        self.labels = labels
        train_transform = transforms.Compose([
            transforms.Resize(size=(128,128)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(15),
            transforms.ToTensor()
        ])
        eval_transform = transforms.Compose([
            transforms.Resize(size=(128,128)),
            transforms.ToTensor()
        ])
        if mode == 'train':
            self.transform = train_transform
        else:
            self.transform = eval_transform
    
    def __len__(self):
        return len(self.paths)
    
    def __getitem__(self, idx):
        img = self.transform(Image.open(self.paths[idx]))
        label = self.labels[idx]
        return img, label
    
    def get_batch(self, indices):  ## indices 是一个包含要获取的图像索引的列表
        images = []
        labels = []
        for idx in indices:    
            img, label = self.__getitem__(idx)
            images.append(img)
            labels.append(label)
        return torch.stack(images), torch.tensor(labels)
    
def get_paths_labels(path):
    def my_key(name):
      return int(name.replace(".jpg",""))+1000000*int(name.split("_")[0])
    imgnames = os.listdir(path)
    imgnames.sort(key=my_key)
    imgpaths = []
    labels = []
    for name in imgnames:
        imgpaths.append(os.path.join(path, name))
        labels.append(int(name.split('_')[0]))
    return imgpaths, labels
train_paths, train_labels = get_paths_labels(args.dataset_dir)

train_set = FoodDataset(train_paths, train_labels, mode='train')

img_indices = [i for i in range(10)]
images, labels = train_set.get_batch(img_indices)
fig, axes = plt.subplots(1, len(img_indices), figsize=(15, 8)) ## 创建一个包含 len(img_indices) 个子图的图形对象，子图的大小为 15x8 英寸。
for i, img in enumerate(images):
        img = img.permute(1, 2, 0).numpy() ## 将图像张量的维度从 (C, H, W) 转换为 (H, W, C)，并将其转换为 NumPy 数组。
        axes[i].imshow(img) ## 在第 i 个子图上显示图像。
        
        
        

: 

# Lime
## 采用遮挡的方式来测那个像素贡献最大

In [ ]:
from skimage.segmentation import slic
import numpy as np
from lime import lime_image

def predict(input):
    model.eval()
    input = torch.FloatTensor(input) ## 将输入数据转换为 PyTorch 的 FloatTensor 类型，以便模型能够处理。
    output = model(input.permute(0, 3, 1, 2).cuda()) ## 将输入图像的维度从 (B, H, W, C) 转换为 (B, C, H, W)，并将其移动到 GPU 上进行计算。
    return output.detach().cpu().numpy() ## 将输出从 GPU 上移动回 CPU，并返回一个 NumPy 数组。

def get_spilt(input):
    return slic(input, n_segments=200, compactness=1, sigma=1) ## 使用 SLIC 算法将输入图像分割成 200 个超像素，compactness 参数控制超像素的紧凑程度，sigma 参数控制平滑程度。

fig, axs = plt.subplots(1, len(img_indices), figsize=(15, 8)) ## 创建一个包含 len(img_indices) 个子图的图形对象，子图的大小为 15x8 英寸。
np.random.seed(16) ## 设置随机种子为 0，以确保结果的可重复性。
for i, (image,label) in enumerate(zip(images.permute(0,2,3,1).numpy(),labels)):
    x = image.astype('double') ## 将图像数据类型转换为 double，以满足 LIME 的输入要求。
    explainer = lime_image.LimeImageExplainer() ## 创建一个 LIME 图像解释器对象。
    explaination = explainer.explain_instance(x, predict, segmentation_fn=get_spilt) ## 使用 LIME 解释器对输入图像进行解释，predict 函数用于生成模型的预测结果，get_spilt 函数用于生成图像的超像素分割。
    lime_img, mask = explaination.get_image_and_mask(label=label.item(),positive_only=False,hide_rest=False,num_features=11, min_weight=0.05) ## 获取解释结果的图像和掩码，label 参数指定要解释的类别，positive_only 参数控制是否只显示正面特征，hide_rest 参数控制是否隐藏其他特征，num_features 参数指定要显示的特征数量，min_weight 参数指定要显示的最小权重。
    axs[i].imshow(lime_img)
    axs[i].set_title(f"Label: {label.item()}")
plt.show()
plt.close()
    


# Saliency Map
## 计算每个像素的梯度，来测那个像素贡献最大

In [ ]:
def normalize(image):
    return (image - image.min()) / (image.max() - image.min()) ## 将输入图像进行归一化处理，使其像素值在 0 和 1 之间。通过减去最小值并除以最大值与最小值的差来实现归一化。

def compute_salence(x, y, model):
    model.eval() 
    x = x.cuda() ## 将输入图像移动到 GPU 上进行计算。
    x = x.requires_grad_() ## 设置输入图像的 requires_grad 属性为 True，以便在反向传播过程中计算输入图像的梯度。
    y_pred = model(x) ## 使用模型对输入图像进行前向传播，得到预测结果。
    loss_func = torch.nn.CrossEntropyLoss()
    loss = loss_func(y_pred, y.cuda()) ## 计算预测结果与真实标签之间的交叉熵损失。
    loss.backward() ## 反向传播计算梯度。
    salences,_ = torch.max(x.grad.data.abs().detach().cpu(), dim=1) ## 计算输入图像的梯度的绝对值，并在通道维度上取最大值，得到每个像素的显著性分数。
    salences = torch.stack([normalize(item) for item in salences]) ## 对每个像素的显著性分数进行归一化处理，使其值在 0 和 1 之间。
    return salences

salences = compute_salence(images, labels, model) ## 计算输入图像的显著性分数。

fig, axs = plt.subplots(2, len(img_indices), figsize=(15, 8)) ## 创建一个包含 2 行 len(img_indices) 列的子图的图形对象，子图的大小为 15x8 英寸。
for row, target in enumerate([images, salences]):
  for column, img in enumerate(target):
    if row==0:
      axs[row][column].imshow(img.permute(1, 2, 0).numpy())
      # What is permute?
      # In pytorch, the meaning of each dimension of image tensor is (channels, height, width)
      # In matplotlib, the meaning of each dimension of image tensor is (height, width, channels)
      # permute is a tool for permuting dimensions of tensors
      # For example, img.permute(1, 2, 0) means that,
      # - 0 dimension is the 1 dimension of the original tensor, which is height
      # - 1 dimension is the 2 dimension of the original tensor, which is width
      # - 2 dimension is the 0 dimension of the original tensor, which is channels
    else:
      axs[row][column].imshow(img.numpy(), cmap=plt.cm.hot)
    
plt.show()
plt.close()

# Smooth Grad
## 随机加入噪声来测那个像素贡献最大

In [ ]:
from torch.autograd import Variable

def normalize(image):
    return (image - image.min()) / (image.max() - image.min()) ## 将输入图像进行归一化处理，使其像素值在 0 和 1 之间。通过减去最小值并除以最大值与最小值的差来实现归一化。

def smooth_grad(x, y, model, epoch, param_sigma_multiplier):
    model.eval()
    mean = 0
    sigma = param_sigma_multiplier / (torch.max(x) - torch.min(x)).item() ## 计算噪声的标准差，param_sigma_multiplier 是一个超参数，用于控制噪声的强度。通过将 param_sigma_multiplier 除以输入图像的最大值与最小值的差来计算标准差。
    smooth = np.zeros(x.cuda().unsqueeze(0).size()) ## 创建一个与输入图像大小相同的全零数组，用于存储平滑梯度的结果,unsqueeze(0) 的作用是为输入图像添加一个新的维度，使其成为一个批次的形式，以便在后续的计算中进行批量处理。
    for i in range(epoch):
        noise = Variable(x.data.new(x.size()).normal_(mean, sigma)) ## 生成与输入图像大小相同的随机噪声，噪声的均值为 mean，标准差为 sigma。
        x_mod = (x + noise).unsqueeze(0).cuda() ## 将输入图像与生成的噪声相加，并添加一个新的维度，使其成为一个批次的形式。
        x_mod = x_mod.requires_grad_() ## 设置修改后的输入图像的 requires_grad 属性为 True，以便在反向传播过程中计算梯度。
        y_pred = model(x_mod) ## 使用模型对修改后的输入图像进行前
        loss_func = torch.nn.CrossEntropyLoss()
        loss = loss_func(y_pred, y.unsqueeze(0).cuda()) ## 计算预测结果与真实标签
        loss.backward() ## 反向传播计算梯度。
        smooth += x_mod.grad.data.abs().cpu().numpy() ## 将计算得到的梯度的绝对值累加到 smooth 数组中。
    smooth = normalize(smooth / epoch) ## 求平均值后对平滑梯度进行归一化处理，使其值在 0 和 1 之间。
    return smooth

smooth = []
for i,l in zip(images, labels): ## image
    smooth.append(smooth_grad(i, l, model, 500, 0.4)) ## 计算输入图像的平滑梯度，epoch 参数指定迭代的次数，param_sigma_multiplier 参数指定噪声的强度。
smooth = np.stack(smooth)
print(smooth.shape)
fig, axs = plt.subplots(2, len(img_indices), figsize=(15, 8)) ## 创建一个包含 2 行 len(img_indices) 列的子图的图形对象，子图的大小为 15x8 英寸。
for row, target in enumerate([images, smooth]):
      for column, img in enumerate(target):
        axs[row][column].imshow(np.transpose(img.reshape(3,128,128),(1,2,0)))

         

# Filter explanation
## 通过卷积核来测那个像素贡献最大
### local: 通过训练图片来看该filter的作用

In [ ]:
def normalize(image):
    return (image - image.min()) / (image.max() - image.min()) 
layer_activation = None

def filter_explanation(x, model, cnnid, filterid, iteration, lr=1):
    model.eval()
    
    def hook(model, input, output): ## 定义一个钩子函数，用于在卷积层进行前向传播时捕获该层的输出。
        global layer_activation ## 声明 layer_activation 变量为全局变量，以便在 hook 函数内部修改它的值。
        layer_activation = output ## 将卷积层的输出保存到 layer_activation 变量中，以便后续使用。
        
    hook_handel = model.cnn[cnnid].register_forward_hook(hook) ## 在模型的第 cnnid 个卷积层上注册一个前向钩子函数，当该层进行前向传播时，hook 函数将被调用。
    model(x.cuda())
    filter_activation = layer_activation[:, filterid, :, :].detach().cpu() ## (B, H, W) 从卷积层的输出中提取出指定 filterid 的激活图，并将其移动回 CPU 上进行处理。
    x = x.cuda()
    x = x.requires_grad_()
    optimizer = optim.Adam([x], lr=lr)
    for iter in range(iteration):
        optimizer.zero_grad()
        model(x)
        objective = -layer_activation[:, filterid, :, :].sum()
        objective.backward()
        optimizer.step()
    filter_visuals = x.detach().cpu().squeeze() ## (C, H, W) 将优化后的输入图像移动回 CPU 上，并去掉批次维度，得到最终的 filter 可视化结果。
    hook_handel.remove() ## 移除之前注册的钩子函数，以避免对后续的前向传播产生影响。
    return filter_activation, filter_visuals

images, labels = train_set.get_batch(img_indices) ## 获取指定索引的图像和标签。
filter_activations, filter_visuals = filter_explanation(images, model, cnnid=6, filterid=0, iteration=100, lr=0.1)
fig, axs = plt.subplots(3, len(img_indices), figsize=(15, 8))
for i, img in enumerate(images):
    axs[0][i].imshow(img.permute(1, 2, 0))
for i, filter_activation in enumerate(filter_activations):
    axs[1][i].imshow(normalize(filter_activation))
for i, filter_visual in enumerate(filter_visuals):
    axs[2][i].imshow(normalize(filter_visual).permute(1, 2, 0))

In [ ]:
images, labels = train_set.get_batch(img_indices) ## 获取指定索引的图像和标签。
filter_activations, filter_visuals = filter_explanation(images, model, cnnid=23, filterid=0, iteration=100, lr=0.1) ## 计算指定卷积层和滤波器的激活图和可视化结果。

fig, axs = plt.subplots(3, len(img_indices), figsize=(15,8))
for i, img in enumerate(images):
    axs[0][i].imshow(img.permute(1, 2, 0))
for i, filter_activation in enumerate(filter_activations):
    axs[1][i].imshow(normalize(filter_activation))
for i, filter_visual in enumerate(filter_visuals):
    axs[2][i].imshow(normalize(filter_visual).permute(1, 2, 0))

# Integrated Gradient
## 通过积分来测那个像素贡献最大

In [ ]:
class IntegratedGradient:
    def __init__(self, model):
        self.model = model
        self.model.eval()
        self.gradients = None
    def generate_image_in_linear_path(self, input, step):
        return [input * i / step for i in range(step)] ## 生成一系列图像，这些图像在输入图像和全零图像之间以线性方式变化，step 参数指定生成的图像数量。
    def generate_gradients(self, input, target_label):
        input.requires_grad = True
        output = self.model(input) ## 使用模型对输入图像进行前向传播，得到预测结果。
        self.model.zero_grad() ## 将模型的梯度清零，以便后续计算新的梯度。
        one_hot_output = torch.FloatTensor(1, output.size()[-1]).zero_().cuda() ## (1, num_classes) 创建一个全零的 one-hot 编码张量，用于指定目标类别。
        one_hot_output[0][target_label] = 1 ## 将目标类别的位置设置为 1，以便在反向传播过程中计算该类别的梯度。
        output.backward(gradient=one_hot_output)
        self.gradients = input.grad ##(B, C, H, W) 将计算得到的梯度保存到 self.gradients 变量中，以便后续使用。
        gradient_as_arr = self.gradients.data.cpu().numpy()[0] ##(c,h,w)
        return gradient_as_arr
    def generate_integrated_gradients(self, input, target_label, step):
        xbar_list = self.generate_image_in_linear_path(input, step) ## 生成一系列图像，这些图像在输入图像和全零图像之间以线性方式变化。
        integrated_gradients = np.zeros(input.size()) ## 创建一个与输入图像大小相同的全零数组，用于存储积分梯度的结果。
        for xbar in xbar_list:
            gradient_as_arr = self.generate_gradients(xbar, target_label) ## 计算每个生成的图像的梯度。
            integrated_gradients += gradient_as_arr/step ## 将每个图像的梯度累加到 integrated_gradients 数组中，并除以 step 参数以求平均值。
        return integrated_gradients[0]
def normalize(image):
    return (image - image.min()) / (image.max() - image.min()) ## 将输入图像进行归一化处理，使其像素值在 0 和 1 之间。通过减去最小值并除以最大值与最小值的差来实现归一化。

images, labels = train_set.get_batch(img_indices) ## 获取指定索引的图像和标签。
images = images.cuda()
ig = IntegratedGradient(model)
integrated_gradients = []
for i,img in enumerate(images):
    img = img.unsqueeze(0) ## (1, C, H, W)
    integrated_gradient = ig.generate_integrated_gradients(img, labels[i], step=10) 
    integrated_gradients.append(integrated_gradient)
fig, axs = plt.subplots(2, len(img_indices), figsize=(15, 8)) 
for i, img in enumerate(images):
    axs[0][i].imshow(img.cpu().permute(1, 2, 0).numpy())
for i, img in enumerate(integrated_gradients):
    axs[1][i].imshow(normalize(img).transpose(1,2,0))
     
        
        